# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing


In [69]:
# Import necessary packages and modules.
import os
import json
import re
import random
import numpy as np
from tqdm import tqdm

import torch
from torch.utils.data import Dataset

from rank_bm25 import BM25Okapi
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

## 1.1 Loading Climate Claim and Evidence Datasets
在本项目中，我们首先加载 climate claim verification 任务所需的所有数据文件，包括：

* training set
* development set
* test set
* baseline predictions
* evidence corpus

其中：

* claim datasets 保存了 claim text、claim label 以及对应的 evidence ids
* evidence.json 保存了所有 evidence passages 的具体文本内容

为了方便后续 retrieval 与 classification，我们首先将所有 JSON 文件统一读取为 Python dictionary 格式，并对数据集规模进行基础统计，同时随机展示部分 evidence examples 以验证数据读取是否正确。

随后，我们进一步对 claim labels 进行 numerical encoding，将字符串标签转换为模型可处理的整数形式：

* SUPPORTS → 0
* REFUTES → 1
* NOT_ENOUGH_INFO → 2
* DISPUTED → 3

最后，我们将原始 claim data 转换为结构化的 claim-level examples。对于 train 和 dev datasets，系统直接使用数据集中提供的 gold evidence ids；而对于 test dataset，由于不存在 ground-truth evidence，系统会自动调用 BM25 retrieval 获取 top-k candidate evidences。

每个 example 会统一保存：

* claim text
* evidence ids
* evidence texts
* concatenated evidence sequence

其中多个 evidence passages 会通过 [SEP] token 拼接，以构建 transformer-based classifier 所需的 claim-evidence representation。

这一 preprocessing step 的主要目标是：

* 统一 train/dev/test 数据格式
* 保持 claim 与 evidence 的对应关系
* 构建模型可直接输入的 structured examples
* 为后续 micro-level classification 提供标准化输入

In [70]:
# =========================
# Generic JSON Loader
# =========================
def load_json(path):
    """
    Load any JSON file from disk.

    Args:
        path (str): file path

    Returns:
        dict: loaded JSON content
    """
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# =========================
# Load All Project Data
# =========================
print("Loading datasets...")

train_claims = load_json("data/train-claims.json")
dev_claims = load_json("data/dev-claims.json")
test_claims = load_json("data/test-claims-unlabelled.json")
baseline_claims = load_json("data/dev-claims-baseline.json")
evidence_dict = load_json("data/evidence.json")


# =========================
# Basic Statistics
# =========================
print("Train claims:", len(train_claims))
print("Dev claims:", len(dev_claims))
print("Test claims:", len(test_claims))
print("Baseline predictions:", len(baseline_claims))
print("Evidence passages:", len(evidence_dict))


# =========================
# Preview Evidence Samples
# =========================
print("\nSample evidence examples:")

for i, (eid, text) in enumerate(evidence_dict.items()):
    print(f"{eid}: {text[:200]}")
    if i >= 2:
        break

Loading datasets...
Train claims: 1228
Dev claims: 154
Test claims: 153
Baseline predictions: 154
Evidence passages: 1208827

Sample evidence examples:
evidence-0: John Bennet Lawes, English entrepreneur and agricultural scientist
evidence-1: Lindberg began his professional career at the age of 16, eventually moving to New York City in 1977.
evidence-2: ``Boston (Ladies of Cambridge)'' by Vampire Weekend


In [71]:
# Mapping from label names to integer IDs
# This is used to convert string labels into numerical format for model training
label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}

# Reverse mapping from IDs back to label names
# This is useful for converting model predictions (integers) back to human-readable labels
id2label = {v: k for k, v in label2id.items()}

In [72]:
def build_examples(claims_dict, labelled=True):
    """
    Build claim-level examples.

    For train/dev:
        Use provided gold evidence IDs.

    For test:
        No labels and often no gold evidence,
        so fallback to BM25 retrieval.
    """
    examples = []

    for claim_id, item in tqdm(claims_dict.items()):
        claim_text = item["claim_text"]

        # ==========================================
        # If gold evidences exist (train/dev)
        # ==========================================
        if "evidences" in item:
            evidence_ids_for_claim = item["evidences"][:5]

            evidence_texts_for_claim = [
                evidence_dict[eid] for eid in evidence_ids_for_claim
            ]

        # ==========================================
        # If no evidences exist (test set)
        # ==========================================
        else:
            evidence_ids_for_claim, evidence_texts_for_claim = retrieve_top_k_evidence(
                claim_text,
                top_k=5
            )

        example = {
            "claim_id": claim_id,
            "claim_text": claim_text,
            "evidence_ids": evidence_ids_for_claim,
            "evidence_texts": evidence_texts_for_claim,
            "evidence_text": " [SEP] ".join(evidence_texts_for_claim)
        }

        if labelled and "claim_label" in item:
            example["label"] = label2id[item["claim_label"]]

        examples.append(example)

    return examples

## 1.2 BM25 Evidence Retrieval
由于 climate claim verification 任务需要从大量 evidence passages 中寻找与 claim 最相关的证据，我们首先构建了一个基于 BM25 的 lexical retrieval system。

具体而言：

* 首先对所有 evidence 文本进行 tokenization
* 将文本统一转为 lowercase
* 仅保留字母与数字 token
* 去除标点与特殊符号

随后，我们使用 BM25Okapi 对整个 evidence corpus 建立检索索引。

该 retrieval module 的作用是：在 test set 没有 gold evidence 时 自动为 claim 检索 top-k relevant evidence passages 为后续 BERT classifier 提供 candidate evidence inputs

In [73]:
import re
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    """
    Tokenizer for BM25.
    Lowercase text and keep only words/numbers.
    """
    text = text.lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    return tokens

# Acquire evidence id and text
evidence_ids = list(evidence_dict.keys())
evidence_texts = list(evidence_dict.values())

print("Tokenizing evidence corpus...")

# Tokenize each evidence document
tokenized_evidence = [
    simple_tokenize(text)
    for text in tqdm(evidence_texts)
]

print("Building BM25 index...")

# Build BM25 model over the tokenized corpus
# This object will be used to compute relevance scores between a query and documents
bm25 = BM25Okapi(tokenized_evidence)

print("BM25 index built.")

Tokenizing evidence corpus...


100%|██████████| 1208827/1208827 [00:03<00:00, 317427.72it/s]


Building BM25 index...
BM25 index built.


### 1.2.1 Semantic Retrieval + Cross-Encoder Reranking

The original notebook uses gold evidence for training and evaluating the classifier. For the final end-to-end system, however, evidence must be retrieved from the evidence corpus. This section adds an embedding-based retriever followed by a cross-encoder reranker. The retriever outputs evidence IDs, which are later passed into the existing Micro--Macro classifier.

The final retrieval configuration can be changed using `RETRIEVAL_TOP_N`, `FINAL_TOP_K`, `FUSION_ALPHA`, and `KEEP_EMBEDDING_TOP`.

In [74]:
from sentence_transformers import SentenceTransformer, CrossEncoder

# =========================
# Improved Hybrid Retrieval + Cross-Encoder Reranking
# =========================
# Main idea:
#   1. Dense embedding retrieval catches semantic matches.
#   2. BM25 retrieval catches lexical/key-term matches.
#   3. The two candidate lists are merged.
#   4. A cross-encoder reranks only the merged candidate pool.
#   5. Final evidence is selected by fixed top-k, not by an unstable per-claim threshold.

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Candidate pool settings
DENSE_TOP_N = 250
BM25_TOP_N = 250
CANDIDATE_POOL_SIZE = 300

# Final evidence settings
FINAL_TOP_K = 5
FUSION_ALPHA = 0.75       # 0.75 = 75% reranker score + 25% first-stage hybrid score
DENSE_WEIGHT = 0.65       # first-stage hybrid dense weight
BM25_WEIGHT = 0.35        # first-stage hybrid BM25 weight

# Backward-compatible aliases used by later cells / optional modules
RETRIEVAL_TOP_N = DENSE_TOP_N
KEEP_EMBEDDING_TOP = 0    # no forced candidate; final ranking decides everything
THRESHOLD = None          # fixed top-k works better than per-claim min-max threshold

print("Loading embedding model...")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Loading cross-encoder reranker...")
reranker = CrossEncoder(RERANKER_MODEL_NAME)

print("Encoding evidence corpus for semantic retrieval...")
evidence_embeddings = embedder.encode(
    evidence_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


def min_max_normalize(scores):
    """Normalize a 1D score array to [0, 1]."""
    scores = np.array(scores, dtype=float)
    if len(scores) == 0:
        return scores
    if scores.max() == scores.min():
        return np.ones_like(scores)
    return (scores - scores.min()) / (scores.max() - scores.min())


def get_top_indices(scores, top_n):
    """Efficiently return indices of the top_n largest scores."""
    scores = np.array(scores)
    top_n = min(top_n, len(scores))

    if top_n <= 0:
        return []

    candidate_idx = np.argpartition(scores, -top_n)[-top_n:]
    candidate_idx = candidate_idx[np.argsort(scores[candidate_idx])[::-1]]
    return candidate_idx.tolist()


def embedding_candidate_retrieve(query, top_n=RETRIEVAL_TOP_N):
    """
    Dense-only retrieval.

    Kept for compatibility with the optional task-specific reranker and oracle diagnostics.

    Returns:
        list of (evidence_id, evidence_text, dense_score)
    """
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    dense_scores = np.dot(evidence_embeddings, query_embedding[0])
    top_indices = get_top_indices(dense_scores, top_n)

    return [
        (evidence_ids[i], evidence_texts[i], float(dense_scores[i]))
        for i in top_indices
    ]


def hybrid_candidate_retrieve(
    query,
    dense_top_n=DENSE_TOP_N,
    bm25_top_n=BM25_TOP_N,
    candidate_pool_size=CANDIDATE_POOL_SIZE,
    dense_weight=DENSE_WEIGHT,
    bm25_weight=BM25_WEIGHT
):
    """
    First-stage hybrid candidate retrieval.

    Uses:
        - dense semantic retrieval
        - BM25 lexical retrieval
        - weighted score fusion

    Returns:
        list of (evidence_id, evidence_text, dense_score, bm25_score, hybrid_score)
    """
    # Dense embedding scores
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    dense_scores = np.dot(evidence_embeddings, query_embedding[0])

    # BM25 lexical scores
    tokenized_query = simple_tokenize(query)
    bm25_scores = np.array(bm25.get_scores(tokenized_query), dtype=float)

    # Normalize over the whole evidence corpus for first-stage fusion
    dense_scores_norm = min_max_normalize(dense_scores)
    bm25_scores_norm = min_max_normalize(bm25_scores)

    # Retrieve candidates from both systems
    dense_top_indices = get_top_indices(dense_scores, dense_top_n)
    bm25_top_indices = get_top_indices(bm25_scores, bm25_top_n)

    # Ordered union: dense candidates first, then BM25 candidates not already present
    candidate_indices = list(dict.fromkeys(dense_top_indices + bm25_top_indices))

    if len(candidate_indices) == 0:
        return []

    candidate_hybrid_scores = (
        dense_weight * dense_scores_norm[candidate_indices]
        + bm25_weight * bm25_scores_norm[candidate_indices]
    )

    sorted_order = np.argsort(candidate_hybrid_scores)[::-1]
    sorted_order = sorted_order[:candidate_pool_size]
    final_candidate_indices = [candidate_indices[i] for i in sorted_order]

    candidates = []
    for idx in final_candidate_indices:
        hybrid_score = (
            dense_weight * dense_scores_norm[idx]
            + bm25_weight * bm25_scores_norm[idx]
        )
        candidates.append(
            (
                evidence_ids[idx],
                evidence_texts[idx],
                float(dense_scores[idx]),
                float(bm25_scores[idx]),
                float(hybrid_score)
            )
        )

    return candidates


def select_final_evidence(reranked, max_final_top_k=FINAL_TOP_K, score_threshold=None, relative_threshold=None):
    """
    Select final evidence from already reranked candidates.

    Default behaviour:
        fixed top-k selection.

    Why not threshold by default?
        Earlier code used min-max normalized scores inside each claim. That makes a fixed
        threshold unstable across claims and caused many zero-hit claims.
    """
    if len(reranked) == 0:
        return []

    selected = []
    selected_ids = set()
    top_score = reranked[0][1]

    for candidate, final_score in reranked:
        # Works for both dense-only tuple and hybrid tuple because eid is always first.
        eid = candidate[0]

        if eid in selected_ids:
            continue

        passes_threshold = True

        if score_threshold is not None:
            passes_threshold = passes_threshold and (final_score >= score_threshold)

        if relative_threshold is not None:
            passes_threshold = passes_threshold and (final_score >= top_score * relative_threshold)

        if passes_threshold:
            selected.append((eid, float(final_score)))
            selected_ids.add(eid)

        if len(selected) >= max_final_top_k:
            break

    # Safety fallback: never return an empty evidence list.
    if len(selected) == 0:
        candidate, final_score = reranked[0]
        selected.append((candidate[0], float(final_score)))

    return selected


def embedding_rerank_retrieve(
    query,
    embedding_top_n=None,
    dense_top_n=DENSE_TOP_N,
    bm25_top_n=BM25_TOP_N,
    candidate_pool_size=CANDIDATE_POOL_SIZE,
    max_final_top_k=FINAL_TOP_K,
    alpha=FUSION_ALPHA,
    keep_embedding_top=KEEP_EMBEDDING_TOP,
    score_threshold=THRESHOLD,
    relative_threshold=None,
    use_hybrid=True
):
    """
    Hybrid candidate retrieval + cross-encoder reranking.

    The parameter `embedding_top_n` is kept as an alias for old cells.
    If supplied, it overrides dense_top_n.
    """
    if embedding_top_n is not None:
        dense_top_n = embedding_top_n

    if use_hybrid:
        candidates = hybrid_candidate_retrieve(
            query,
            dense_top_n=dense_top_n,
            bm25_top_n=bm25_top_n,
            candidate_pool_size=candidate_pool_size
        )
        first_stage_scores = [hybrid_score for eid, text, dense_score, bm25_score, hybrid_score in candidates]
        pairs = [[query, text] for eid, text, dense_score, bm25_score, hybrid_score in candidates]
    else:
        candidates = embedding_candidate_retrieve(query, top_n=dense_top_n)
        first_stage_scores = [dense_score for eid, text, dense_score in candidates]
        pairs = [[query, text] for eid, text, dense_score in candidates]

    if len(candidates) == 0:
        return []

    rerank_scores = reranker.predict(
        pairs,
        batch_size=32,
        show_progress_bar=False
    )

    rerank_scores_norm = min_max_normalize(rerank_scores)
    first_stage_scores_norm = min_max_normalize(first_stage_scores)

    final_scores = (
        alpha * rerank_scores_norm
        + (1 - alpha) * first_stage_scores_norm
    )

    reranked = sorted(
        zip(candidates, final_scores),
        key=lambda x: x[1],
        reverse=True
    )

    # `keep_embedding_top` is preserved only for compatibility. It is usually better as 0.
    if keep_embedding_top and use_hybrid:
        dense_fallback = embedding_candidate_retrieve(query, top_n=keep_embedding_top)
        fallback_ids = [eid for eid, text, score in dense_fallback]

        selected = []
        selected_ids = set()
        score_lookup = {candidate[0]: float(score) for candidate, score in reranked}

        for eid in fallback_ids:
            if eid not in selected_ids:
                selected.append((eid, score_lookup.get(eid, 0.0)))
                selected_ids.add(eid)

        for candidate, final_score in reranked:
            eid = candidate[0]
            if eid in selected_ids:
                continue
            selected.append((eid, float(final_score)))
            selected_ids.add(eid)
            if len(selected) >= max_final_top_k:
                break

        return selected[:max_final_top_k]

    return select_final_evidence(
        reranked,
        max_final_top_k=max_final_top_k,
        score_threshold=score_threshold,
        relative_threshold=relative_threshold
    )


def retrieve_top_k_evidence(claim_text, top_k=FINAL_TOP_K):
    """
    Compatibility wrapper used by build_examples() for unlabelled claims.

    Returns:
        evidence_ids_for_claim, evidence_texts_for_claim
    """
    retrieved = embedding_rerank_retrieve(
        claim_text,
        dense_top_n=DENSE_TOP_N,
        bm25_top_n=BM25_TOP_N,
        candidate_pool_size=CANDIDATE_POOL_SIZE,
        max_final_top_k=top_k,
        alpha=FUSION_ALPHA,
        score_threshold=THRESHOLD,
        relative_threshold=None,
        use_hybrid=True
    )

    retrieved_ids = [eid for eid, score in retrieved]
    retrieved_texts = [evidence_dict[eid] for eid in retrieved_ids]

    return retrieved_ids, retrieved_texts


def generate_embedding_reranker_predictions(
    claims,
    embedding_top_n=None,
    dense_top_n=DENSE_TOP_N,
    bm25_top_n=BM25_TOP_N,
    candidate_pool_size=CANDIDATE_POOL_SIZE,
    max_final_top_k=FINAL_TOP_K,
    alpha=FUSION_ALPHA,
    keep_embedding_top=KEEP_EMBEDDING_TOP,
    score_threshold=THRESHOLD,
    relative_threshold=None,
    default_label="NOT_ENOUGH_INFO",
    use_hybrid=True
):
    """
    Generate retrieval-only predictions.

    claim_label is a placeholder and will be replaced by the Micro--Macro classifier later.
    """
    if embedding_top_n is not None:
        dense_top_n = embedding_top_n

    predictions = {}

    for claim_id, item in tqdm(claims.items(), desc="Hybrid retrieval + reranker"):
        claim_text = item["claim_text"]

        retrieved = embedding_rerank_retrieve(
            claim_text,
            dense_top_n=dense_top_n,
            bm25_top_n=bm25_top_n,
            candidate_pool_size=candidate_pool_size,
            max_final_top_k=max_final_top_k,
            alpha=alpha,
            keep_embedding_top=keep_embedding_top,
            score_threshold=score_threshold,
            relative_threshold=relative_threshold,
            use_hybrid=use_hybrid
        )

        predictions[claim_id] = {
            "claim_label": default_label,
            "evidences": [eid for eid, score in retrieved]
        }

    return predictions


def evaluate_retrieval(predictions, groundtruth, name="Retrieval"):
    """
    Diagnostic evaluation for evidence retrieval only.
    This mirrors the evidence F-score component of eval.py, but also reports
    precision, recall, and zero-hit ratio.
    """
    precisions = []
    recalls = []
    f1s = []
    zero_hit = 0
    missing_predictions = 0

    for claim_id, gold_item in groundtruth.items():
        if claim_id not in predictions:
            missing_predictions += 1
            precisions.append(0.0)
            recalls.append(0.0)
            f1s.append(0.0)
            zero_hit += 1
            continue

        gold = set(gold_item.get("evidences", []))
        pred = set(predictions[claim_id].get("evidences", []))

        hit = len(gold & pred)
        precision = hit / len(pred) if len(pred) > 0 else 0.0
        recall = hit / len(gold) if len(gold) > 0 else 0.0
        f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

        if hit == 0:
            zero_hit += 1

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    result = {
        "name": name,
        "precision": float(np.mean(precisions)),
        "recall": float(np.mean(recalls)),
        "f1": float(np.mean(f1s)),
        "zero_hit_claims": zero_hit,
        "zero_hit_ratio": zero_hit / len(groundtruth),
        "missing_predictions": missing_predictions
    }

    print(f"===== {name} =====")
    print(f"Retrieval Precision: {result['precision']:.4f}")
    print(f"Retrieval Recall:    {result['recall']:.4f}")
    print(f"Retrieval F1:        {result['f1']:.4f}")
    print(f"Zero-hit claims:     {zero_hit} / {len(groundtruth)}")
    print(f"Zero-hit ratio:      {result['zero_hit_ratio']:.4f}")
    if missing_predictions:
        print(f"Missing predictions: {missing_predictions}")

    return result


def evaluate_candidate_pool_recall(
    claims,
    dense_top_n=DENSE_TOP_N,
    bm25_top_n=BM25_TOP_N,
    candidate_pool_size=CANDIDATE_POOL_SIZE
):
    """
    Check whether gold evidence appears inside the candidate pool before reranking.

    If this is high but final retrieval is low, the reranker/final selection is the bottleneck.
    If this is low, the first-stage retriever is the bottleneck.
    """
    recalls = []
    zero_hit = 0

    for claim_id, item in tqdm(claims.items(), desc="Candidate pool recall"):
        claim_text = item["claim_text"]
        gold = set(item.get("evidences", []))

        candidates = hybrid_candidate_retrieve(
            claim_text,
            dense_top_n=dense_top_n,
            bm25_top_n=bm25_top_n,
            candidate_pool_size=candidate_pool_size
        )

        candidate_ids = set([eid for eid, text, dense_score, bm25_score, hybrid_score in candidates])
        hit = len(gold & candidate_ids)

        recall = hit / len(gold) if len(gold) > 0 else 0.0
        recalls.append(recall)

        if hit == 0:
            zero_hit += 1

    result = {
        "candidate_recall": float(np.mean(recalls)),
        "zero_hit_claims": zero_hit,
        "zero_hit_ratio": zero_hit / len(claims)
    }

    print("===== Candidate Pool Diagnostic =====")
    print(f"Candidate Recall: {result['candidate_recall']:.4f}")
    print(f"Zero-hit claims:  {zero_hit} / {len(claims)}")
    print(f"Zero-hit ratio:   {result['zero_hit_ratio']:.4f}")

    return result


def oracle_retrieval_from_hybrid_candidates(
    claims,
    dense_top_n=DENSE_TOP_N,
    bm25_top_n=BM25_TOP_N,
    candidate_pool_size=CANDIDATE_POOL_SIZE,
    final_top_k=FINAL_TOP_K
):
    """
    Oracle diagnostic: if a gold evidence appears in the hybrid candidate pool,
    select it first, then fill remaining slots with top hybrid candidates.

    This gives an upper bound for reranking/final selection under the current candidate pool.
    """
    oracle_preds = {}

    for claim_id, item in tqdm(claims.items(), desc="Hybrid candidate oracle"):
        claim_text = item["claim_text"]
        gold = set(item.get("evidences", []))

        candidates = hybrid_candidate_retrieve(
            claim_text,
            dense_top_n=dense_top_n,
            bm25_top_n=bm25_top_n,
            candidate_pool_size=candidate_pool_size
        )
        candidate_ids = [eid for eid, text, dense_score, bm25_score, hybrid_score in candidates]

        selected = []

        for eid in candidate_ids:
            if eid in gold and eid not in selected:
                selected.append(eid)
            if len(selected) >= final_top_k:
                break

        for eid in candidate_ids:
            if eid not in selected:
                selected.append(eid)
            if len(selected) >= final_top_k:
                break

        oracle_preds[claim_id] = {
            "claim_label": item.get("claim_label", "NOT_ENOUGH_INFO"),
            "evidences": selected
        }

    return oracle_preds


def run_retrieval_grid_search(
    claims,
    dense_values=(200, 300, 500),
    bm25_values=(100, 200, 300),
    pool_values=(200, 300, 500),
    top_k_values=(3, 5, 7),
    alpha_values=(0.6, 0.7, 0.8, 0.9),
    save_path="retrieval_grid_results.json"
):
    """
    Optional grid search for retrieval parameters.
    This is slow because each setting reruns cross-encoder reranking.
    """
    grid_results = []

    for dense_n in dense_values:
        for bm25_n in bm25_values:
            for pool_size in pool_values:
                for top_k in top_k_values:
                    for alpha in alpha_values:
                        run_name = (
                            f"dense={dense_n}, bm25={bm25_n}, "
                            f"pool={pool_size}, k={top_k}, alpha={alpha}"
                        )
                        print("\nRunning", run_name)

                        preds = generate_embedding_reranker_predictions(
                            claims,
                            dense_top_n=dense_n,
                            bm25_top_n=bm25_n,
                            candidate_pool_size=pool_size,
                            max_final_top_k=top_k,
                            alpha=alpha,
                            keep_embedding_top=0,
                            score_threshold=None,
                            default_label="NOT_ENOUGH_INFO",
                            use_hybrid=True
                        )

                        result = evaluate_retrieval(
                            preds,
                            claims,
                            name=run_name
                        )

                        grid_results.append({
                            "dense_top_n": dense_n,
                            "bm25_top_n": bm25_n,
                            "candidate_pool_size": pool_size,
                            "top_k": top_k,
                            "alpha": alpha,
                            "precision": result["precision"],
                            "recall": result["recall"],
                            "f1": result["f1"],
                            "zero_hit_ratio": result["zero_hit_ratio"]
                        })

                        with open(save_path, "w", encoding="utf-8") as f:
                            json.dump(grid_results, f, indent=2)

    grid_results_sorted = sorted(grid_results, key=lambda x: x["f1"], reverse=True)

    print("\n===== Top 10 retrieval settings =====")
    for row in grid_results_sorted[:10]:
        print(row)

    return grid_results_sorted


def build_examples_from_retrieval_predictions(claims_dict, retrieval_predictions, labelled=True):
    """
    Convert retrieval predictions into the same example format expected by the
    existing Micro--Macro classifier.

    This is the bridge:
        retrieved evidence IDs -> evidence texts -> micro classifier -> macro classifier
    """
    examples = []

    for claim_id, item in claims_dict.items():
        claim_text = item["claim_text"]
        evidence_ids_for_claim = retrieval_predictions[claim_id]["evidences"]
        evidence_texts_for_claim = [evidence_dict[eid] for eid in evidence_ids_for_claim]

        example = {
            "claim_id": claim_id,
            "claim_text": claim_text,
            "evidence_ids": evidence_ids_for_claim,
            "evidence_texts": evidence_texts_for_claim,
            "evidence_text": " [SEP] ".join(evidence_texts_for_claim)
        }

        if labelled and "claim_label" in item:
            example["label"] = label2id[item["claim_label"]]

        examples.append(example)

    return examples


Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading cross-encoder reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Encoding evidence corpus for semantic retrieval...


Batches:   0%|          | 0/18888 [00:00<?, ?it/s]

## 1.2.2 Optional Task-specific Cross-Encoder Reranker

This optional module trains a task-specific cross-encoder reranker using the training set. Positive pairs are constructed from claim + gold evidence, while hard negative pairs are sampled from high-ranking embedding candidates that are not gold evidence. The trained reranker can replace the off-the-shelf MS MARCO cross-encoder in the final retrieval stage.

By default this block is switched off to keep the notebook fast and reproducible. Set `RUN_TASK_SPECIFIC_RERANKER = True` and `USE_TASK_SPECIFIC_RERANKER_FOR_FINAL = True` to train and use it.

In [102]:
# =========================
# Optional task-specific reranker configuration
# =========================
# Keep these False for the standard/best current pipeline.
# Turn both True if you want to train and use the task-specific reranker.
RUN_TASK_SPECIFIC_RERANKER = False
USE_TASK_SPECIFIC_RERANKER_FOR_FINAL = False

TASK_RERANKER_MODEL_NAME = "distilbert-base-uncased"
TASK_RERANKER_CANDIDATE_TOP_N = 50
TASK_RERANKER_NEG_PER_CLAIM = 6
TASK_RERANKER_EPOCHS = 2
TASK_RERANKER_LR = 2e-5
TASK_RERANKER_MAX_LENGTH = 256

# Convenience alias; the earlier cells load evidence into evidence_dict.
EVIDENCE_DICT = evidence_dict

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def build_task_reranker_pairs(
    claims,
    evidence_dict,
    candidate_top_n=TASK_RERANKER_CANDIDATE_TOP_N,
    neg_per_claim=TASK_RERANKER_NEG_PER_CLAIM,
    seed=42
):
    """
    Build binary training pairs for a task-specific reranker.

    Positive examples:
        claim + gold evidence -> 1

    Hard negative examples:
        claim + high-ranking embedding candidate that is not gold -> 0
    """
    random.seed(seed)

    claim_texts = []
    evidence_texts = []
    labels = []

    for claim_id, item in tqdm(claims.items(), desc="Building task reranker pairs"):
        claim_text = item["claim_text"]
        gold_ids = set(item.get("evidences", []))

        # Positive pairs: claim + gold evidence
        for eid in gold_ids:
            if eid in evidence_dict:
                claim_texts.append(claim_text)
                evidence_texts.append(evidence_dict[eid])
                labels.append(1)

        # Hard negatives: embedding candidates that are not gold
        candidates = embedding_candidate_retrieve(
            claim_text,
            top_n=candidate_top_n
        )

        hard_negative_ids = [
            eid for eid, text, score in candidates
            if eid not in gold_ids
        ][:neg_per_claim]

        for eid in hard_negative_ids:
            if eid in evidence_dict:
                claim_texts.append(claim_text)
                evidence_texts.append(evidence_dict[eid])
                labels.append(0)

    return claim_texts, evidence_texts, labels


class TaskRerankerPairDataset(Dataset):
    """Dataset for binary claim-evidence relevance classification."""

    def __init__(self, claim_texts, evidence_texts, labels, tokenizer, max_length=TASK_RERANKER_MAX_LENGTH):
        self.claim_texts = claim_texts
        self.evidence_texts = evidence_texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.claim_texts[idx],
            self.evidence_texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

        if "token_type_ids" in encoding:
            item["token_type_ids"] = encoding["token_type_ids"].squeeze(0)

        return item


def compute_task_reranker_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# These variables are populated only if RUN_TASK_SPECIFIC_RERANKER is True.
task_reranker_tokenizer = None
task_reranker_model = None
task_reranker_trainer = None

if RUN_TASK_SPECIFIC_RERANKER:
    print("Preparing task-specific reranker training pairs...")

    task_train_q, task_train_e, task_train_y = build_task_reranker_pairs(
        train_claims_small,
        EVIDENCE_DICT,
        candidate_top_n=TASK_RERANKER_CANDIDATE_TOP_N,
        neg_per_claim=TASK_RERANKER_NEG_PER_CLAIM
    )

    task_dev_q, task_dev_e, task_dev_y = build_task_reranker_pairs(
        dev_claims_small,
        EVIDENCE_DICT,
        candidate_top_n=TASK_RERANKER_CANDIDATE_TOP_N,
        neg_per_claim=TASK_RERANKER_NEG_PER_CLAIM
    )

    print("Task reranker train pairs:", len(task_train_y))
    print("Task reranker dev pairs:", len(task_dev_y))
    print("Train positive ratio:", sum(task_train_y) / len(task_train_y))
    print("Dev positive ratio:", sum(task_dev_y) / len(task_dev_y))

    task_reranker_tokenizer = AutoTokenizer.from_pretrained(TASK_RERANKER_MODEL_NAME)

    task_train_dataset = TaskRerankerPairDataset(
        task_train_q,
        task_train_e,
        task_train_y,
        task_reranker_tokenizer,
        max_length=TASK_RERANKER_MAX_LENGTH
    )

    task_dev_dataset = TaskRerankerPairDataset(
        task_dev_q,
        task_dev_e,
        task_dev_y,
        task_reranker_tokenizer,
        max_length=TASK_RERANKER_MAX_LENGTH
    )

    task_reranker_model = AutoModelForSequenceClassification.from_pretrained(
        TASK_RERANKER_MODEL_NAME,
        num_labels=2
    )

    task_reranker_training_args = TrainingArguments(
        output_dir="./task_specific_reranker",
        learning_rate=TASK_RERANKER_LR,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        num_train_epochs=TASK_RERANKER_EPOCHS,
        weight_decay=0.01,
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        dataloader_pin_memory=False
    )

    try:
        task_reranker_trainer = Trainer(
            model=task_reranker_model,
            args=task_reranker_training_args,
            train_dataset=task_train_dataset,
            eval_dataset=task_dev_dataset,
            processing_class=task_reranker_tokenizer,
            compute_metrics=compute_task_reranker_metrics
        )
    except TypeError:
        task_reranker_trainer = Trainer(
            model=task_reranker_model,
            args=task_reranker_training_args,
            train_dataset=task_train_dataset,
            eval_dataset=task_dev_dataset,
            compute_metrics=compute_task_reranker_metrics
        )

    task_reranker_trainer.train()
    print("Task-specific reranker pair-level dev evaluation:")
    print(task_reranker_trainer.evaluate())
else:
    print("Task-specific reranker training skipped. Set RUN_TASK_SPECIFIC_RERANKER = True to train it.")


def _get_active_task_reranker():
    """Return the trained task-specific reranker objects, or raise a clear error."""
    if task_reranker_model is None or task_reranker_tokenizer is None:
        raise RuntimeError(
            "Task-specific reranker is not trained/loaded. "
            "Set RUN_TASK_SPECIFIC_RERANKER=True and run this cell before using it."
        )
    return task_reranker_model, task_reranker_tokenizer


def predict_task_reranker_scores(
    query,
    candidates,
    model=None,
    tokenizer=None,
    max_length=TASK_RERANKER_MAX_LENGTH,
    batch_size=32
):
    """Score candidate evidence passages using the trained task-specific reranker."""
    if model is None or tokenizer is None:
        model, tokenizer = _get_active_task_reranker()

    model.eval()

    scoring_device = torch.device(
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    )

    model.to(scoring_device)
    scores = []

    for start in range(0, len(candidates), batch_size):
        batch = candidates[start:start + batch_size]

        batch_claims = [query] * len(batch)
        batch_evidences = [text for eid, text, emb_score in batch]

        encoding = tokenizer(
            batch_claims,
            batch_evidences,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoding = {k: v.to(scoring_device) for k, v in encoding.items()}

        with torch.no_grad():
            outputs = model(**encoding)
            probs = torch.softmax(outputs.logits, dim=1)
            relevant_scores = probs[:, 1].detach().cpu().numpy()

        scores.extend(relevant_scores.tolist())

    return scores


def task_specific_embedding_rerank_retrieve(
    query,
    embedding_top_n=RETRIEVAL_TOP_N,
    max_final_top_k=FINAL_TOP_K,
    alpha=FUSION_ALPHA,
    keep_embedding_top=KEEP_EMBEDDING_TOP,
    score_threshold=None,
    relative_threshold=None
):
    """
    Retrieval using embedding candidates + trained task-specific reranker.

    It mirrors embedding_rerank_retrieve(), but replaces the off-the-shelf
    MS MARCO cross-encoder scores with task-specific relevance scores.
    """
    candidates = embedding_candidate_retrieve(
        query,
        top_n=embedding_top_n
    )

    task_scores = predict_task_reranker_scores(
        query=query,
        candidates=candidates,
        model=task_reranker_model,
        tokenizer=task_reranker_tokenizer,
        max_length=TASK_RERANKER_MAX_LENGTH,
        batch_size=32
    )

    emb_scores = [emb_score for eid, text, emb_score in candidates]

    task_scores_norm = min_max_normalize(task_scores)
    emb_scores_norm = min_max_normalize(emb_scores)

    final_scores = (
        alpha * task_scores_norm
        + (1 - alpha) * emb_scores_norm
    )

    reranked = sorted(
        zip(candidates, final_scores),
        key=lambda x: x[1],
        reverse=True
    )

    selected = []
    selected_ids = set()

    # Keep the best embedding candidate as a fallback.
    for eid, text, emb_score in candidates[:keep_embedding_top]:
        selected.append((eid, float(emb_score)))
        selected_ids.add(eid)

    # Fixed top-k by default. Optional absolute/relative thresholds can be enabled.
    top_score = reranked[0][1]

    for (eid, text, emb_score), final_score in reranked:
        if eid in selected_ids:
            continue

        passes_threshold = True

        if score_threshold is not None:
            passes_threshold = passes_threshold and (final_score >= score_threshold)

        if relative_threshold is not None:
            passes_threshold = passes_threshold and (final_score >= top_score * relative_threshold)

        if passes_threshold:
            selected.append((eid, float(final_score)))
            selected_ids.add(eid)

        if len(selected) >= max_final_top_k:
            break

    # Ensure at least one evidence passage.
    if len(selected) == 0:
        (eid, text, emb_score), final_score = reranked[0]
        selected.append((eid, float(final_score)))

    return selected


def generate_task_specific_reranker_predictions(
    claims,
    embedding_top_n=RETRIEVAL_TOP_N,
    max_final_top_k=FINAL_TOP_K,
    alpha=FUSION_ALPHA,
    keep_embedding_top=KEEP_EMBEDDING_TOP,
    score_threshold=None,
    relative_threshold=None,
    default_label="NOT_ENOUGH_INFO"
):
    """Generate retrieval predictions using the trained task-specific reranker."""
    predictions = {}

    for claim_id, item in tqdm(claims.items(), desc="Task-specific reranker retrieval"):
        claim_text = item["claim_text"]

        retrieved = task_specific_embedding_rerank_retrieve(
            claim_text,
            embedding_top_n=embedding_top_n,
            max_final_top_k=max_final_top_k,
            alpha=alpha,
            keep_embedding_top=keep_embedding_top,
            score_threshold=score_threshold,
            relative_threshold=relative_threshold
        )

        predictions[claim_id] = {
            "claim_label": default_label,
            "evidences": [eid for eid, score in retrieved]
        }

    return predictions


# Optional retrieval-only test for the trained task-specific reranker.
if RUN_TASK_SPECIFIC_RERANKER:
    dev_task_reranker_retrieval_predictions = generate_task_specific_reranker_predictions(
        dev_claims_small,
        embedding_top_n=RETRIEVAL_TOP_N,
        max_final_top_k=FINAL_TOP_K,
        alpha=FUSION_ALPHA,
        keep_embedding_top=KEEP_EMBEDDING_TOP,
        score_threshold=THRESHOLD,
        default_label="NOT_ENOUGH_INFO"
    )

    evaluate_retrieval(
        dev_task_reranker_retrieval_predictions,
        dev_claims_small,
        name="Task-specific reranker retrieval"
    )


Task-specific reranker training skipped. Set RUN_TASK_SPECIFIC_RERANKER = True to train it.


## 1.3 Constructing Model-ready Claim–Evidence Inputs
在完成 preprocessing pipeline 后，我们进一步构建 train、development 与 test datasets 对应的 model-ready claim-evidence examples。

对于 train 与 dev datasets，系统会保留 gold claim labels 并直接使用数据集中提供的 evidence passages；而对于 test dataset，系统仅构建 claim-evidence inputs，不包含 ground-truth labels。

最终生成的 examples 会统一包含：

* claim text
* evidence ids
* evidence texts
* concatenated evidence sequence

这些 structured examples 将作为后续 BERT-based micro classifier 的输入。

In [76]:
train_claims_small = dict(list(train_claims.items()))
dev_claims_small = dict(list(dev_claims.items()))
test_claims_small = dict(list(test_claims.items()))

# Train/dev use gold evidence here for training the micro and macro classifiers.
# Test examples are intentionally not built at this stage, because test has no gold evidence
# and running retrieval here would be slow and unnecessary. Test predictions are generated
# in the final end-to-end section using the retrieval module.
train_examples = build_examples(
    train_claims_small,
    labelled=True
)

dev_examples = build_examples(
    dev_claims_small,
    labelled=True
)

test_examples = None

print("Train examples:", len(train_examples))
print("Dev examples:", len(dev_examples))
print("Test examples: built later with retrieved evidence")

print("Sample processed example:")
print(train_examples[0])

100%|██████████| 154/154 [00:00<00:00, 144598.79it/s]

Train examples: 1228
Dev examples: 154
Test examples: built later with retrieved evidence
Sample processed example:
{'claim_id': 'claim-1937', 'claim_text': 'Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.', 'evidence_ids': ['evidence-442946', 'evidence-1194317', 'evidence-12171'], 'evidence_texts': ['At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal life, so raising the concentration to 10,000 ppm (1%) or higher for several hours will eliminate pests such as whiteflies and spider mites in a greenhouse.', 'Plants can grow as much as 50 percent faster in concentrations of 1,000 ppm CO 2 when compared with ambient conditions, though this assumes no change in climate and no limitation on other nutrients.', 'Higher carbon dioxide concentrations will favourably affect plant growth and demand for water.'], 'evidence_text':

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)




## 2.1 Micro-level Classifer

### 2.1.1 Building Sentence-level Micro Examples

In [77]:
micro_label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2
}

micro_id2label = {v: k for k, v in micro_label2id.items()}

In [78]:
def build_micro_examples(examples, labelled=True, skip_disputed=True):
    """
    Convert claim-level examples into sentence-level micro examples.

    Each claim with provided evidence becomes multiple micro examples:
        claim + evidence_i -> micro label

    DISPUTED is skipped during micro training because it is a macro-level label.
    """
    micro_examples = []

    for ex in examples:
        claim_id = ex["claim_id"]
        claim_text = ex["claim_text"]

        evidence_ids = ex["evidence_ids"]
        evidence_texts = ex["evidence_texts"]

        if labelled:
            macro_label = id2label[ex["label"]]

            if skip_disputed and macro_label == "DISPUTED":
                continue

            if macro_label not in micro_label2id:
                continue

            micro_label = micro_label2id[macro_label]

        for rank, (eid, ev_text) in enumerate(zip(evidence_ids, evidence_texts)):
            item = {
                "claim_id": claim_id,
                "claim_text": claim_text,
                "evidence_id": eid,
                "evidence_text": ev_text,
                "rank": rank
            }

            if labelled:
                item["label"] = micro_label

            micro_examples.append(item)

    return micro_examples


train_micro_examples = build_micro_examples(
    train_examples,
    labelled=True,
    skip_disputed=True
)

dev_micro_examples = build_micro_examples(
    dev_examples,
    labelled=True,
    skip_disputed=True
)

print("Train micro examples:", len(train_micro_examples))
print("Dev micro examples:", len(dev_micro_examples))
print(train_micro_examples[0])

Train micro examples: 3730
Dev micro examples: 433
{'claim_id': 'claim-126', 'claim_text': 'El Niño drove record highs in global temperatures suggesting rise may not be down to man-made emissions.', 'evidence_id': 'evidence-338219', 'evidence_text': 'While ‘climate change’ can be due to natural forces or human activity, there is now substantial evidence to indicate that human activity – and specifically increased greenhouse gas (GHGs) emissions – is a key factor in the pace and extent of global temperature increases.', 'rank': 0, 'label': 1}


### 2.1.2 BERT-based Micro Classifier
在 micro-level classification 阶段，我们采用 bert-base-uncased 作为基础 transformer encoder，用于完成 sentence-level claim-evidence entailment prediction。对于每一个 micro example，系统会将 claim text + evidence text 作为 sentence pair 输入到 BERT tokenizer 中。随后，tokenizer 会自动生成：

* input_ids
* attention_mask
* token_type_ids

等 transformer 所需的输入 tensors。其中：

* claim_text 被视为 sentence A
* evidence_text 被视为 sentence B

从而使 BERT 能够学习 claim 与 evidence 之间的语义关系。

为了适配 batch training，我们采用：

* padding="max_length"
* truncation=True

并将 maximum sequence length 设置为 256，以统一输入长度并降低 GPU memory consumption。随后，我们进一步构建 MicroClaimEvidenceDataset，将所有 sentence-level micro examples 转换为 PyTorch Dataset 格式，方便后续 HuggingFace Trainer 进行 fine-tuning。

最后，我们使用 AutoModelForSequenceClassification 构建 BERT-based micro classifier，并输出三个 logits，对应：
* SUPPORTS
* REFUTES
* NOT_ENOUGH_INFO

从而完成 sentence-level entailment prediction。

In [79]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 256
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 1

In [80]:
class MicroClaimEvidenceDataset(Dataset):
    """
    Dataset for sentence-level BERT micro-verdict prediction.

    Input:
        claim_text + one evidence sentence

    Output:
        SUPPORTS / REFUTES / NOT_ENOUGH_INFO
    """

    def __init__(self, micro_examples, tokenizer, max_length=256, labelled=True):
        self.examples = micro_examples
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labelled = labelled

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]

        encoding = self.tokenizer(
            ex["claim_text"],
            ex["evidence_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
        }

        if "token_type_ids" in encoding:
            item["token_type_ids"] = encoding["token_type_ids"].squeeze(0)

        if self.labelled:
            item["labels"] = torch.tensor(ex["label"], dtype=torch.long)

        return item
train_micro_dataset = MicroClaimEvidenceDataset(
    train_micro_examples,
    tokenizer,
    max_length=MAX_LENGTH,
    labelled=True
)

dev_micro_dataset = MicroClaimEvidenceDataset(
    dev_micro_examples,
    tokenizer,
    max_length=MAX_LENGTH,
    labelled=True
)

print("Train micro dataset:", len(train_micro_dataset))
print("Dev micro dataset:", len(dev_micro_dataset))

Train micro dataset: 3730
Dev micro dataset: 433


In [81]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=micro_id2label,
    label2id=micro_label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 2.1.3 Training Strategy and Evaluation Metrics
在训练阶段，我们首先考虑 micro-level labels 存在类别不平衡的问题。由于不同 entailment labels 的样本数量差异较大，如果直接使用 standard cross-entropy loss，模型可能会过度偏向 majority class。因此，我们根据训练集中各类别样本数量计算 class weights，并将其加入 weighted cross-entropy loss 中，以增强模型对 minority labels 的学习能力。

随后，我们进一步扩展 HuggingFace Trainer，实现 WeightedLossTrainer。该 trainer 会在 loss computation 阶段自动使用预先计算好的 class weights，从而缓解 label imbalance 对训练稳定性的影响。

在 evaluation 部分，我们使用：

* accuracy
* macro F1
* weighted F1

作为主要评估指标。其中：

* accuracy 用于衡量整体预测正确率
* macro F1 更关注不同类别之间的 balanced performance
* weighted F1 会考虑类别样本数量差异

由于本任务存在类别不平衡问题，因此我们主要使用macro F1 作为 selecting best model 的核心指标。

最后，我们使用 HuggingFace TrainingArguments 配置训练过程，包括：
* learning rate
* batch size
* number of epochs
* checkpoint saving strategy
* evaluation strategy

并通过 Trainer 对 micro-level BERT classifier 进行 fine-tuning。训练完成后，系统会自动在 development set 上进行 evaluation，并输出最终的 micro-level classification performance。

In [82]:
from collections import Counter
import torch
import numpy as np

train_label_counts = Counter([ex["label"] for ex in train_micro_examples])

num_classes = len(micro_label2id)
total_count = sum(train_label_counts.values())

class_weights = []

for i in range(num_classes):
    weight = total_count / (num_classes * train_label_counts[i])
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class weights:", class_weights)

Class weights: tensor([0.9258, 2.7206, 0.6442])


In [83]:
from transformers import Trainer
import torch.nn as nn

class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )

        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [84]:
from sklearn.metrics import accuracy_score, f1_score

def compute_micro_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    return {
        "micro_accuracy": accuracy,
        "micro_macro_f1": macro_f1,
        "micro_weighted_f1": weighted_f1
    }

In [85]:
training_args = TrainingArguments(
    output_dir="./micro_macro_bert",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,

    logging_dir="./logs",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="micro_macro_f1",#"micro_accuracy",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none"
)
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_micro_dataset,
    eval_dataset=dev_micro_dataset,
    compute_metrics=compute_micro_metrics,
    class_weights=class_weights
)
trainer.train()
micro_dev_results = trainer.evaluate()
print(micro_dev_results)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Micro Accuracy,Micro Macro F1,Micro Weighted F1
1,0.834502,0.943260,0.561201,0.503212,0.554823


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Micro Accuracy,Micro Macro F1,Micro Weighted F1
0.834502,0.943260,1,0.561201,0.503212,0.554823


{'eval_loss': 0.9432604312896729, 'eval_micro_accuracy': 0.5612009237875288, 'eval_micro_macro_f1': 0.5032123010571009, 'eval_micro_weighted_f1': 0.5548225546200967}


## 2.2 Macro-level Classifer

### 2.2.1 Building Macro-level Features from Micro Predictions

在完成 micro-level classifier 训练后，我们进一步构建 macro-level aggregation features。Macro-level classifier 的目标不是判断单条 evidence 与 claim 的关系，而是综合多个 evidence-level predictions，预测最终的 claim-level label——SUPPORTS / REFUTES / NOT_ENOUGH_INFO / DISPUTED。

首先，我们使用已经训练好的 micro BERT classifier 对每个 claim 对应的多条 evidence 分别进行预测，并提取每条 evidence 的三分类概率分布：P(SUPPORTS), P(REFUTES), P(NOT_ENOUGH_INFO)，因此，每个 claim 会被表示为一个 micro prediction matrix：[num_evidence, 3]，其中每一行对应一条 evidence 的 micro-level prediction probabilities。

为了帮助 macro model 判断 evidence 之间是否存在冲突，我们进一步计算 uncertainty-related features，例如 Jensen-Shannon divergence。该特征用于衡量多条 evidence predictions 之间的分歧程度。如果不同 evidence 同时表现出较强的 SUPPORTS 与 REFUTES signals，则该 claim 更可能属于 DISPUTED 类别。

随后，我们将每个 claim 的 micro probabilities、uncertainty features、evidence ids 和 macro label 封装成 macro-level feature examples。由于不同 claim 的 evidence 数量可能不同，我们在 MacroAggregationDataset 中对 evidence 维度进行 padding 或 truncation，并使用 evidence_mask 区分真实 evidence 与 padding 部分。

最后，这些 features 被转换为 PyTorch Dataset 和 DataLoader，用于后续 macro-level aggregation model 的训练。

In [86]:
macro_label2id = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}

macro_id2label = {v: k for k, v in macro_label2id.items()}

def entropy_np(probs):
    probs = np.array(probs, dtype=np.float32)
    return -np.sum(probs * np.log(probs + 1e-8))


def js_divergence_np(prob_list):
    prob_array = np.array(prob_list, dtype=np.float32)
    mean_prob = prob_array.mean(axis=0)
    return entropy_np(mean_prob) - np.mean([entropy_np(p) for p in prob_array])


def support_refute_variance_np(prob_list):
    prob_array = np.array(prob_list, dtype=np.float32)
    return np.var(prob_array[:, 0]) + np.var(prob_array[:, 1])
def get_micro_probs_for_claim(claim_text, evidence_texts):
    """
    Use trained micro BERT to predict probs for all evidences of one claim.
    Output shape: [num_evidence, 3]
    """
    model.eval()
    all_probs = []

    for ev_text in evidence_texts:
        encoding = tokenizer(
            claim_text,
            ev_text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoding = {k: v.to(model.device) for k, v in encoding.items()}

        with torch.no_grad():
            outputs = model(**encoding)
            temperature = 1
            probs = torch.softmax(outputs.logits/temperature, dim=-1).squeeze(0)

        all_probs.append(probs.cpu().numpy())

    return np.array(all_probs, dtype=np.float32)
def build_macro_features(examples, labelled=True):
    """
    Build claim-level features for learned attention aggregation.

    Each claim becomes:
        micro_probs: [num_evidence, 3]
        uncertainty_features: [JSD, support_refute_variance]
        label: 4-class macro label
    """
    feature_examples = []

    for ex in examples:
        claim_text = ex["claim_text"]
        evidence_texts = ex["evidence_texts"]

        micro_probs = get_micro_probs_for_claim(claim_text, evidence_texts)

        jsd = js_divergence_np(micro_probs)
        sr_var = support_refute_variance_np(micro_probs)

        item = {
            "claim_id": ex["claim_id"],
            "evidence_ids": ex["evidence_ids"],
            "micro_probs": micro_probs,
            "uncertainty_features": np.array([jsd], dtype=np.float32)
        }

        if labelled:
            macro_label = id2label[ex["label"]]
            item["label"] = macro_label2id[macro_label]

        feature_examples.append(item)

    return feature_examples
train_macro_features = build_macro_features(train_examples, labelled=True)
dev_macro_features = build_macro_features(dev_examples, labelled=True)

print("Train macro features:", len(train_macro_features))
print("Dev macro features:", len(dev_macro_features))
print(train_macro_features[0].keys())

Train macro features: 1228
Dev macro features: 154
dict_keys(['claim_id', 'evidence_ids', 'micro_probs', 'uncertainty_features', 'label'])


In [87]:
from torch.utils.data import DataLoader
class MacroAggregationDataset(Dataset):
    def __init__(self, feature_examples, labelled=True, max_evidence=5):
        self.examples = feature_examples
        self.labelled = labelled
        self.max_evidence = max_evidence

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]

        micro_probs = torch.tensor(ex["micro_probs"], dtype=torch.float32)

        # pad / truncate to fixed number of evidence
        num_evidence = micro_probs.shape[0]

        if num_evidence > self.max_evidence:
            micro_probs = micro_probs[:self.max_evidence]
            evidence_mask = torch.ones(self.max_evidence, dtype=torch.float32)

        else:
            pad_size = self.max_evidence - num_evidence
            pad = torch.zeros(pad_size, micro_probs.shape[1])
            micro_probs = torch.cat([micro_probs, pad], dim=0)

            evidence_mask = torch.cat([
                torch.ones(num_evidence, dtype=torch.float32),
                torch.zeros(pad_size, dtype=torch.float32)
            ])

        item = {
            "claim_id": ex["claim_id"],
            "evidence_ids": ex["evidence_ids"],
            "micro_probs": micro_probs,
            "evidence_mask": evidence_mask,
            "uncertainty_features": torch.tensor(ex["uncertainty_features"], dtype=torch.float32)
        }

        if self.labelled:
            item["labels"] = torch.tensor(ex["label"], dtype=torch.long)

        return item
def macro_collate_fn(batch):
    output = {
        "claim_ids": [x["claim_id"] for x in batch],
        "evidence_ids": [x["evidence_ids"] for x in batch],
        "micro_probs": torch.stack([x["micro_probs"] for x in batch]),
        "evidence_mask": torch.stack([x["evidence_mask"] for x in batch]),
        "uncertainty_features": torch.stack([x["uncertainty_features"] for x in batch]),
    }

    if "labels" in batch[0]:
        output["labels"] = torch.stack([x["labels"] for x in batch])

    return output

train_macro_dataset = MacroAggregationDataset(train_macro_features, labelled=True)
dev_macro_dataset = MacroAggregationDataset(dev_macro_features, labelled=True)

train_macro_loader = DataLoader(
    train_macro_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=macro_collate_fn
)

dev_macro_loader = DataLoader(
    dev_macro_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=macro_collate_fn
)

### 2.2.2 Attention-based Macro Aggregator Training
在 macro-level model 中，我们采用 learned attention aggregation 的方式对多条 evidence 的 micro-level prediction probabilities 进行加权汇总。相比简单平均或 majority voting，attention mechanism 可以让模型自动学习哪些 evidence 对最终 claim-level prediction 更重要。

具体而言，模型首先对每条 evidence 的 micro probability vector 计算 attention score，并通过 softmax 得到 normalized attention weights。随后，模型根据 attention weights 对所有 evidence-level probabilities 进行加权求和，得到一个 claim-level weighted probability representation。

接着，我们将该 weighted probability representation 与 uncertainty features 拼接，作为最终 macro classifier 的输入。该 classifier 输出四个 logits，对应：SUPPORTS / REFUTES / NOT_ENOUGH_INFO / DISPUTED。

其中，DISPUTED 类别需要依赖多个 evidence 之间的综合关系，因此更适合在 macro-level 阶段预测，而不是在 micro-level classifier 中直接预测。

训练阶段，我们使用 AdamW optimizer 对 macro aggregator 进行训练，并在每个 epoch 后在 development set 上计算 macro-level accuracy。通过这一过程，macro model 能够学习如何从多个 sentence-level predictions 中聚合出最终的 claim-level decision。

In [88]:
import torch.nn as nn

macro_hidden_size = 64
macro_dropout = 0.5
macro_lr = 5e-3
macro_weight_decay = 0.01
uncertainty_dim = 1

class LearnedAttentionMacroAggregator(nn.Module):
    def __init__(
        self,
        num_micro_labels=3,
        uncertainty_dim=1,
        hidden_size=32,
        dropout=0.1,
        num_macro_labels=4
    ):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(num_micro_labels, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(num_micro_labels + uncertainty_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_macro_labels)
        )

    def forward(self, micro_probs, uncertainty_features, evidence_mask=None, labels=None):
        attn_logits = self.attention(micro_probs).squeeze(-1)

        if evidence_mask is not None:
            attn_logits = attn_logits.masked_fill(evidence_mask == 0, -1e9)

        attn_weights = torch.softmax(attn_logits, dim=1)

        weighted_probs = torch.sum(
            micro_probs * attn_weights.unsqueeze(-1),
            dim=1
        )

        features = torch.cat([weighted_probs, uncertainty_features], dim=-1)
        logits = self.classifier(features)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return {
            "loss": loss,
            "logits": logits,
            "attention_weights": attn_weights
        }
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

macro_model = LearnedAttentionMacroAggregator(
    num_micro_labels=3,
    uncertainty_dim=uncertainty_dim,
    hidden_size=macro_hidden_size,
    dropout=macro_dropout,
    num_macro_labels=4
).to(device)

optimizer = torch.optim.AdamW(
    macro_model.parameters(),
    lr=macro_lr,
    weight_decay=macro_weight_decay
)

In [89]:
def train_macro_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in loader:
        optimizer.zero_grad()

        micro_probs = batch["micro_probs"].to(device)
        uncertainty_features = batch["uncertainty_features"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            micro_probs=micro_probs,
            uncertainty_features=uncertainty_features,
            labels=labels
        )

        loss = outputs["loss"]
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate_macro_model(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            micro_probs = batch["micro_probs"].to(device)
            uncertainty_features = batch["uncertainty_features"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                micro_probs=micro_probs,
                uncertainty_features=uncertainty_features
            )

            preds = torch.argmax(outputs["logits"], dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.mean(np.array(all_preds) == np.array(all_labels))

MACRO_EPOCHS = 20

for epoch in range(MACRO_EPOCHS):
    train_loss = train_macro_one_epoch(
        macro_model,
        train_macro_loader,
        optimizer,
        device
    )

    dev_acc = evaluate_macro_model(
        macro_model,
        dev_macro_loader,
        device
    )

    print(f"Epoch {epoch + 1}/{MACRO_EPOCHS}")
    print("Macro train loss:", train_loss)
    print("Macro dev accuracy:", dev_acc)

Epoch 1/20
Macro train loss: 0.936624606514906
Macro dev accuracy: 0.6493506493506493
Epoch 2/20
Macro train loss: 0.7251925602942318
Macro dev accuracy: 0.6428571428571429
Epoch 3/20
Macro train loss: 0.6584906311971801
Macro dev accuracy: 0.6883116883116883
Epoch 4/20
Macro train loss: 0.6108642102932775
Macro dev accuracy: 0.6818181818181818
Epoch 5/20
Macro train loss: 0.6029770323014879
Macro dev accuracy: 0.6818181818181818
Epoch 6/20
Macro train loss: 0.5866667174480178
Macro dev accuracy: 0.6948051948051948
Epoch 7/20
Macro train loss: 0.5842672782381634
Macro dev accuracy: 0.6948051948051948
Epoch 8/20
Macro train loss: 0.5820790954514757
Macro dev accuracy: 0.7012987012987013
Epoch 9/20
Macro train loss: 0.5569064554746275
Macro dev accuracy: 0.7272727272727273
Epoch 10/20
Macro train loss: 0.5648976414428128
Macro dev accuracy: 0.7012987012987013
Epoch 11/20
Macro train loss: 0.5490747434752328
Macro dev accuracy: 0.7077922077922078
Epoch 12/20
Macro train loss: 0.5507795143

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 3.1 End-to-end Claim Verification Inference
在完成 micro-level classifier 与 macro-level aggregator 的训练后，我们进一步对整个 Micro–Macro framework 进行 end-to-end inference。

对于每个 claim：

系统首先使用 micro-level BERT classifier 对所有 evidence passages 分别生成 sentence-level entailment probabilities
随后，macro-level attention aggregator 会综合多个 micro predictions
并输出最终的 claim-level prediction

最终系统能够生成：
claim
→ final claim label
→ supporting evidence ids

其中最终 claim labels 包括：SUPPORTS / REFUTES / NOT_ENOUGH_INFO/ DISPUTED

随后，我们将 prediction results 保存为 JSON 文件，并使用课程提供的 evaluation script 对 development set 进行整体评估。

这一过程能够验证整个 Micro–Macro pipeline 在 climate claim verification task 上的 end-to-end performance。

In [90]:
def predict_macro_with_learned_attention(model, loader, device):
    model.eval()
    predictions = {}

    with torch.no_grad():
        for batch in loader:
            micro_probs = batch["micro_probs"].to(device)
            uncertainty_features = batch["uncertainty_features"].to(device)

            outputs = model(
                micro_probs=micro_probs,
                uncertainty_features=uncertainty_features
            )

            preds = torch.argmax(outputs["logits"], dim=-1).cpu().numpy()

            for claim_id, evidence_ids, pred_id in zip(
                batch["claim_ids"],
                batch["evidence_ids"],
                preds
            ):
                predictions[claim_id] = {
                    "claim_label": macro_id2label[int(pred_id)],
                    "evidences": evidence_ids
                }

    return predictions

import json

dev_predictions = predict_macro_with_learned_attention(
    macro_model,
    dev_macro_loader,
    device
)

with open("dev_predictions_learned_attention_macro.json", "w", encoding="utf-8") as f:
    json.dump(dev_predictions, f, indent=2)

with open("dev_claims_small.json", "w", encoding="utf-8") as f:
    json.dump(dev_claims_small, f, indent=2)

!python eval.py \
    --predictions dev_predictions_learned_attention_macro.json \
    --groundtruth dev_claims_small.json

Evidence Retrieval F-score (F)    = 1.0
Claim Classification Accuracy (A) = 0.6948051948051948
Harmonic Mean of F and A          = 0.8199233716475095


## 3.2 Micro-level Classification Evaluation

为了进一步分析 micro-level classifier 的 sentence-level entailment prediction performance，我们对 development set 上的所有 claim-evidence pairs 进行 evaluation。

具体而言，系统会分别计算：

* precision
* recall
* F1-score

并生成：

* classification report
* confusion matrix

用于分析不同 micro labels 的分类效果。

其中：

* SUPPORTS
* REFUTES
* NOT_ENOUGH_INFO

分别对应 sentence-level evidence relations。

通过 confusion matrix，我们能够进一步观察模型容易混淆的类别。例如，部分 NOT_ENOUGH_INFO examples 可能会被错误分类为 SUPPORTS 或 REFUTES，反映出 evidence ambiguity 对 entailment prediction 带来的挑战。

In [91]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def evaluate_micro_per_class(model, dataset, tokenizer=None):
    model.eval()

    all_preds = []
    all_labels = []

    for i in range(len(dataset)):
        item = dataset[i]

        inputs = {
            "input_ids": item["input_ids"].unsqueeze(0).to(model.device),
            "attention_mask": item["attention_mask"].unsqueeze(0).to(model.device),
        }

        if "token_type_ids" in item:
            inputs["token_type_ids"] = item["token_type_ids"].unsqueeze(0).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=-1).item()

        all_preds.append(pred)
        all_labels.append(item["labels"].item())

    print("Micro-level classification report:")
    print(
        classification_report(
            all_labels,
            all_preds,
            target_names=[
                "SUPPORTS",
                "REFUTES",
                "NOT_ENOUGH_INFO"
            ],
            digits=4,
            zero_division=0
        )
    )

    print("Confusion matrix:")
    print(confusion_matrix(all_labels, all_preds))

    return all_labels, all_preds
micro_true, micro_pred = evaluate_micro_per_class(
    model,
    dev_micro_dataset
)

Micro-level classification report:
                 precision    recall  f1-score   support

       SUPPORTS     0.5773    0.7427    0.6496       171
        REFUTES     0.3214    0.3158    0.3186        57
NOT_ENOUGH_INFO     0.6242    0.4780    0.5414       205

       accuracy                         0.5612       433
      macro avg     0.5076    0.5122    0.5032       433
   weighted avg     0.5658    0.5612    0.5548       433

Confusion matrix:
[[127   5  39]
 [ 19  18  20]
 [ 74  33  98]]


## 3.3 Macro-level Classification Evaluation

除了 micro-level evaluation 外，我们还进一步对最终的 macro-level claim predictions 进行整体评估。

具体而言，系统会将：

* predicted macro labels
* ground-truth claim labels

进行比较，并计算：

* precision
* recall
* F1-score
* confusion matrix

其中 macro-level labels 包括：

* SUPPORTS
* REFUTES
* NOT_ENOUGH_INFO
* DISPUTED

相比 micro-level classification，macro-level evaluation 更关注多个 evidence predictions 的综合结果，因此 DISPUTED 类别的预测能力能够反映 learned attention aggregation 是否成功学习 evidence-level conflicts。

通过这一 evaluation，我们能够整体分析 Micro–Macro framework 在 climate claim verification task 上的最终性能表现。

In [92]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_macro_predictions(predictions, groundtruth_claims):
    y_true = []
    y_pred = []

    for claim_id, pred_item in predictions.items():
        if claim_id not in groundtruth_claims:
            continue

        true_label = groundtruth_claims[claim_id]["claim_label"]
        pred_label = pred_item["claim_label"]

        y_true.append(true_label)
        y_pred.append(pred_label)

    labels = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]

    print("Macro-level classification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=labels,
            digits=4,
            zero_division=0
        )
    )

    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=labels))

    return y_true, y_pred
macro_true, macro_pred = evaluate_macro_predictions(
    dev_predictions,
    dev_claims
)

Macro-level classification report:
                 precision    recall  f1-score   support

       SUPPORTS     0.6337    0.9412    0.7574        68
        REFUTES     0.8000    0.2963    0.4324        27
NOT_ENOUGH_INFO     0.8140    0.8537    0.8333        41
       DISPUTED     0.0000    0.0000    0.0000        18

       accuracy                         0.6948       154
      macro avg     0.5619    0.5228    0.5058       154
   weighted avg     0.6368    0.6948    0.6321       154

Confusion matrix:
[[64  0  4  0]
 [17  8  2  0]
 [ 6  0 35  0]
 [14  2  2  0]]


## 3.4 Final End-to-End System: Retrieved Evidence + Micro--Macro Classifier

This section combines the two components into one complete system:

1. The semantic retriever + cross-encoder reranker retrieves final evidence IDs.
2. The retrieved evidence texts are passed into the trained BERT micro classifier.
3. The learned-attention macro aggregator predicts the final claim label.
4. The output JSON contains both `claim_label` and `evidences`, so it can be evaluated by `eval.py`.

This is the final setting that should be used for development-set reporting and test-set prediction generation.

In [103]:
# =========================
# Final dev-set retrieval
# =========================
# Recommended default:
#   Hybrid candidate retrieval + off-the-shelf cross-encoder reranker
#   Fixed top-k selection, no absolute threshold.

# Optional: first check whether gold evidence is inside the candidate pool.
candidate_pool_diagnostic = evaluate_candidate_pool_recall(
    dev_claims_small,
    dense_top_n=DENSE_TOP_N,
    bm25_top_n=BM25_TOP_N,
    candidate_pool_size=CANDIDATE_POOL_SIZE
)

if USE_TASK_SPECIFIC_RERANKER_FOR_FINAL:
    print("Using task-specific reranker for final retrieval.")
    print("===== Current retrieval config =====")
    print("DENSE_TOP_N =", DENSE_TOP_N)
    print("BM25_TOP_N =", BM25_TOP_N)
    print("CANDIDATE_POOL_SIZE =", CANDIDATE_POOL_SIZE)
    print("FINAL_TOP_K =", FINAL_TOP_K)
    print("FUSION_ALPHA =", FUSION_ALPHA)
    print("THRESHOLD =", THRESHOLD)
    print("KEEP_EMBEDDING_TOP =", KEEP_EMBEDDING_TOP)
    print("hybrid_candidate_retrieve =", hybrid_candidate_retrieve.__name__)
    final_dev_retrieval_predictions = generate_task_specific_reranker_predictions(
        dev_claims_small,
        embedding_top_n=RETRIEVAL_TOP_N,
        max_final_top_k=FINAL_TOP_K,
        alpha=FUSION_ALPHA,
        keep_embedding_top=0,
        score_threshold=None,
        default_label="NOT_ENOUGH_INFO"
    )
else:
    print("Using hybrid retrieval + off-the-shelf cross-encoder reranker.")
    final_dev_retrieval_predictions = generate_embedding_reranker_predictions(
        dev_claims_small,
        dense_top_n=DENSE_TOP_N,
        bm25_top_n=BM25_TOP_N,
        candidate_pool_size=CANDIDATE_POOL_SIZE,
        max_final_top_k=FINAL_TOP_K,
        alpha=FUSION_ALPHA,
        keep_embedding_top=0,
        score_threshold=None,
        relative_threshold=None,
        default_label="NOT_ENOUGH_INFO",
        use_hybrid=True
    )

# Retrieval-only diagnostic before classification.
retrieval_diagnostic = evaluate_retrieval(
    final_dev_retrieval_predictions,
    dev_claims_small,
    name="Final retrieval before Micro--Macro classification"
)

# =========================
# Bridge retrieval output into existing Micro--Macro classifier
# =========================
final_dev_retrieved_examples = build_examples_from_retrieval_predictions(
    dev_claims_small,
    final_dev_retrieval_predictions,
    labelled=True
)

final_dev_macro_features = build_macro_features(
    final_dev_retrieved_examples,
    labelled=True
)

final_dev_macro_dataset = MacroAggregationDataset(
    final_dev_macro_features,
    labelled=True,
    max_evidence=FINAL_TOP_K
)

final_dev_macro_loader = DataLoader(
    final_dev_macro_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=macro_collate_fn
)

# =========================
# Final end-to-end prediction
# =========================
final_dev_predictions = predict_macro_with_learned_attention(
    macro_model,
    final_dev_macro_loader,
    device
)

final_dev_output_path = "dev_final_retrieval_micro_macro_predictions.json"
with open(final_dev_output_path, "w", encoding="utf-8") as f:
    json.dump(final_dev_predictions, f, indent=2)

print("Saved final dev predictions to:", final_dev_output_path)

# Official evaluation: evidence retrieval F-score, claim classification accuracy, harmonic mean.
import subprocess

subprocess.run([
    "python",
    "eval.py",
    "--predictions",
    "dev_final_retrieval_micro_macro_predictions.json",
    "--groundtruth",
    "dev_claims_small.json"
], check=False)

# Additional diagnostics
final_retrieval_diagnostic = evaluate_retrieval(
    final_dev_predictions,
    dev_claims_small,
    name="Final system retrieval"
)

final_macro_true, final_macro_pred = evaluate_macro_predictions(
    final_dev_predictions,
    dev_claims_small
)


Candidate pool recall: 100%|██████████| 154/154 [12:49<00:00,  5.00s/it]


===== Candidate Pool Diagnostic =====
Candidate Recall: 0.8104
Zero-hit claims:  9 / 154
Zero-hit ratio:   0.0584
Using hybrid retrieval + off-the-shelf cross-encoder reranker.


Hybrid retrieval + reranker: 100%|██████████| 154/154 [15:13<00:00,  5.93s/it]


===== Final retrieval before Micro--Macro classification =====
Retrieval Precision: 0.1727
Retrieval Recall:    0.3092
Retrieval F1:        0.2066
Zero-hit claims:     67 / 154
Zero-hit ratio:      0.4351
Saved final dev predictions to: dev_final_retrieval_micro_macro_predictions.json
Evidence Retrieval F-score (F)    = 0.2066429602143888
Claim Classification Accuracy (A) = 0.487012987012987
Harmonic Mean of F and A          = 0.2901663445732036
===== Final system retrieval =====
Retrieval Precision: 0.1727
Retrieval Recall:    0.3092
Retrieval F1:        0.2066
Zero-hit claims:     67 / 154
Zero-hit ratio:      0.4351
Macro-level classification report:
                 precision    recall  f1-score   support

       SUPPORTS     0.5429    0.5588    0.5507        68
        REFUTES     1.0000    0.2963    0.4571        27
NOT_ENOUGH_INFO     0.3816    0.7073    0.4957        41
       DISPUTED     0.0000    0.0000    0.0000        18

       accuracy                         0.4870     

## 3.5 Optional Test-Set Prediction Generation

Run this cell only when you are ready to generate the final unlabelled test predictions. The test output file follows the expected format: each claim has a predicted `claim_label` and a list of retrieved `evidences`.

In [94]:
# =========================
# Optional: generate test-set predictions for submission
# =========================
RUN_TEST_PREDICTION = False

if RUN_TEST_PREDICTION:
    if USE_TASK_SPECIFIC_RERANKER_FOR_FINAL:
        print("Using task-specific reranker for test retrieval.")
        test_retrieval_predictions = generate_task_specific_reranker_predictions(
            test_claims_small,
            embedding_top_n=RETRIEVAL_TOP_N,
            max_final_top_k=FINAL_TOP_K,
            alpha=FUSION_ALPHA,
            keep_embedding_top=0,
            score_threshold=None,
            default_label="NOT_ENOUGH_INFO"
        )
    else:
        print("Using hybrid retrieval + off-the-shelf cross-encoder reranker for test retrieval.")
        test_retrieval_predictions = generate_embedding_reranker_predictions(
            test_claims_small,
            dense_top_n=DENSE_TOP_N,
            bm25_top_n=BM25_TOP_N,
            candidate_pool_size=CANDIDATE_POOL_SIZE,
            max_final_top_k=FINAL_TOP_K,
            alpha=FUSION_ALPHA,
            keep_embedding_top=0,
            score_threshold=None,
            relative_threshold=None,
            default_label="NOT_ENOUGH_INFO",
            use_hybrid=True
        )

    test_retrieved_examples = build_examples_from_retrieval_predictions(
        test_claims_small,
        test_retrieval_predictions,
        labelled=False
    )

    test_macro_features = build_macro_features(
        test_retrieved_examples,
        labelled=False
    )

    test_macro_dataset = MacroAggregationDataset(
        test_macro_features,
        labelled=False,
        max_evidence=FINAL_TOP_K
    )

    test_macro_loader = DataLoader(
        test_macro_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=macro_collate_fn
    )

    final_test_predictions = predict_macro_with_learned_attention(
        macro_model,
        test_macro_loader,
        device
    )

    test_output_path = "test-claims-predictions.json"
    with open(test_output_path, "w", encoding="utf-8") as f:
        json.dump(final_test_predictions, f, indent=2)

    print("Saved final test predictions to:", test_output_path)
    print("Number of test predictions:", len(final_test_predictions))
else:
    print("Set RUN_TEST_PREDICTION = True when you are ready to generate test predictions.")


Set RUN_TEST_PREDICTION = True when you are ready to generate test predictions.


In [95]:
torch.save(macro_model.state_dict(), "macro_model.pt")

## Optional: Save Trained Macro Aggregator Weights

These cells save the learned-attention macro aggregator locally. They do not upload anything externally.

In [96]:
torch.save({
    "model_state_dict": macro_model.state_dict(),

    "macro_hidden_size": macro_hidden_size,
    "macro_dropout": macro_dropout,
    "macro_lr": macro_lr,
    "macro_weight_decay": macro_weight_decay,

    "uncertainty_dim": uncertainty_dim,

    "num_micro_labels": 3,
    "num_macro_labels": 4,

    "macro_label2id": macro_label2id,
    "macro_id2label": macro_id2label

}, "macro_aggregator.pt")

In [97]:
import os

print(os.path.exists("macro_aggregator.pt"))

True


In [98]:
# =========================
# Retrieval Oracle and Optional Grid Search Diagnostics
# =========================
# These diagnostics are useful for understanding whether the bottleneck is:
#   1. first-stage candidate retrieval, or
#   2. reranking/final selection.

# Dense-only oracle: old diagnostic, kept for comparison.
def oracle_retrieval_from_dense_candidates(claims, top_n=200, final_top_k=5):
    oracle_preds = {}

    for claim_id, item in tqdm(claims.items(), desc="Dense candidate oracle"):
        claim_text = item["claim_text"]
        gold = set(item.get("evidences", []))

        candidates = embedding_candidate_retrieve(claim_text, top_n=top_n)
        candidate_ids = [eid for eid, text, score in candidates]

        selected = []

        for eid in candidate_ids:
            if eid in gold and eid not in selected:
                selected.append(eid)
            if len(selected) >= final_top_k:
                break

        for eid in candidate_ids:
            if eid not in selected:
                selected.append(eid)
            if len(selected) >= final_top_k:
                break

        oracle_preds[claim_id] = {
            "claim_label": item.get("claim_label", "NOT_ENOUGH_INFO"),
            "evidences": selected
        }

    return oracle_preds


# 1. Dense-only oracle
oracle_dense_preds = oracle_retrieval_from_dense_candidates(
    dev_claims_small,
    top_n=200,
    final_top_k=FINAL_TOP_K
)

evaluate_retrieval(
    oracle_dense_preds,
    dev_claims_small,
    name="Oracle selection from dense top 200"
)

# 2. Hybrid candidate pool recall
candidate_pool_diagnostic = evaluate_candidate_pool_recall(
    dev_claims_small,
    dense_top_n=DENSE_TOP_N,
    bm25_top_n=BM25_TOP_N,
    candidate_pool_size=CANDIDATE_POOL_SIZE
)

# 3. Hybrid oracle
oracle_hybrid_preds = oracle_retrieval_from_hybrid_candidates(
    dev_claims_small,
    dense_top_n=DENSE_TOP_N,
    bm25_top_n=BM25_TOP_N,
    candidate_pool_size=CANDIDATE_POOL_SIZE,
    final_top_k=FINAL_TOP_K
)

evaluate_retrieval(
    oracle_hybrid_preds,
    dev_claims_small,
    name="Oracle selection from hybrid candidate pool"
)

# 4. Optional grid search. This is slow, so keep it False unless tuning.
RUN_RETRIEVAL_GRID_SEARCH = True

if RUN_RETRIEVAL_GRID_SEARCH:
    grid_results_sorted = run_retrieval_grid_search(
        dev_claims_small,
        dense_values=(200, 300, 500),
        bm25_values=(100, 200, 300),
        pool_values=(200, 300, 500),
        top_k_values=(3, 5, 7),
        alpha_values=(0.6, 0.7, 0.8, 0.9),
        save_path="retrieval_grid_results.json"
    )
else:
    print("Set RUN_RETRIEVAL_GRID_SEARCH = True if you want to tune retrieval parameters.")


Dense candidate oracle: 100%|██████████| 154/154 [00:09<00:00, 16.97it/s]


===== Oracle selection from dense top 200 =====
Retrieval Precision: 0.4229
Retrieval Recall:    0.7047
Retrieval F1:        0.4973
Zero-hit claims:     13 / 154
Zero-hit ratio:      0.0844


Candidate pool recall: 100%|██████████| 154/154 [06:37<00:00,  2.58s/it]


===== Candidate Pool Diagnostic =====
Candidate Recall: 0.8104
Zero-hit claims:  9 / 154
Zero-hit ratio:   0.0584


Hybrid candidate oracle:  46%|████▌     | 71/154 [03:15<03:48,  2.75s/it]


KeyboardInterrupt: 

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*

## 3.6 Harmonic Mean-Oriented Tuning

The official score is not retrieval F1 alone. It is the harmonic mean between evidence retrieval F-score and claim classification accuracy.

This diagnostic block tunes the full end-to-end system by running retrieval, rebuilding macro features from retrieved evidence, predicting labels, and calculating the final harmonic mean. It is slower than retrieval-only tuning, so it is disabled by default.


In [101]:
# =========================
# Optional: Harmonic Mean-Oriented End-to-End Tuning
# =========================
# Official objective:
#   harmonic = 2 * retrieval_f1 * classification_accuracy / (retrieval_f1 + classification_accuracy)
#
# Why this matters:
#   A setting with slightly lower retrieval F1 can still win if it gives cleaner evidence
#   and improves classification accuracy. So after retrieval tuning, tune a small set of
#   candidate settings by the final harmonic score.

from sklearn.metrics import accuracy_score


def compute_classification_accuracy(predictions, groundtruth_claims):
    y_true = []
    y_pred = []

    for claim_id, gold_item in groundtruth_claims.items():
        if claim_id not in predictions:
            continue
        y_true.append(gold_item["claim_label"])
        y_pred.append(predictions[claim_id]["claim_label"])

    if len(y_true) == 0:
        return 0.0

    return float(accuracy_score(y_true, y_pred))


def compute_harmonic_score(retrieval_f1, classification_accuracy):
    if retrieval_f1 + classification_accuracy == 0:
        return 0.0
    return float(2 * retrieval_f1 * classification_accuracy / (retrieval_f1 + classification_accuracy))


def run_end_to_end_setting(
    claims,
    dense_top_n=250,
    bm25_top_n=250,
    candidate_pool_size=300,
    final_top_k=5,
    alpha=0.75,
    dense_weight=DENSE_WEIGHT,
    bm25_weight=BM25_WEIGHT,
    save_predictions_path=None
):
    """
    Run one complete retrieval + classification setting and return official-style metrics.
    """
    global DENSE_WEIGHT, BM25_WEIGHT

    old_dense_weight = DENSE_WEIGHT
    old_bm25_weight = BM25_WEIGHT

    DENSE_WEIGHT = dense_weight
    BM25_WEIGHT = bm25_weight

    try:
        retrieval_predictions = generate_embedding_reranker_predictions(
            claims,
            dense_top_n=dense_top_n,
            bm25_top_n=bm25_top_n,
            candidate_pool_size=candidate_pool_size,
            max_final_top_k=final_top_k,
            alpha=alpha,
            keep_embedding_top=0,
            score_threshold=None,
            relative_threshold=None,
            default_label="NOT_ENOUGH_INFO",
            use_hybrid=True
        )

        retrieval_metrics = evaluate_retrieval(
            retrieval_predictions,
            claims,
            name=(
                f"End-to-end setting: dense={dense_top_n}, bm25={bm25_top_n}, "
                f"pool={candidate_pool_size}, k={final_top_k}, alpha={alpha}, "
                f"dense_w={dense_weight}, bm25_w={bm25_weight}"
            )
        )

        retrieved_examples = build_examples_from_retrieval_predictions(
            claims,
            retrieval_predictions,
            labelled=True
        )

        macro_features = build_macro_features(
            retrieved_examples,
            labelled=True
        )

        macro_dataset = MacroAggregationDataset(
            macro_features,
            labelled=True,
            max_evidence=final_top_k
        )

        macro_loader = DataLoader(
            macro_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=macro_collate_fn
        )

        final_predictions = predict_macro_with_learned_attention(
            macro_model,
            macro_loader,
            device
        )

        classification_accuracy = compute_classification_accuracy(
            final_predictions,
            claims
        )

        harmonic = compute_harmonic_score(
            retrieval_metrics["f1"],
            classification_accuracy
        )

        result = {
            "dense_top_n": dense_top_n,
            "bm25_top_n": bm25_top_n,
            "candidate_pool_size": candidate_pool_size,
            "final_top_k": final_top_k,
            "alpha": alpha,
            "dense_weight": dense_weight,
            "bm25_weight": bm25_weight,
            "retrieval_precision": retrieval_metrics["precision"],
            "retrieval_recall": retrieval_metrics["recall"],
            "retrieval_f1": retrieval_metrics["f1"],
            "zero_hit_ratio": retrieval_metrics["zero_hit_ratio"],
            "classification_accuracy": classification_accuracy,
            "harmonic": harmonic
        }

        print("===== Official-style End-to-End Metrics =====")
        print(f"Evidence Retrieval F-score (F)    = {retrieval_metrics['f1']}")
        print(f"Claim Classification Accuracy (A) = {classification_accuracy}")
        print(f"Harmonic Mean of F and A          = {harmonic}")

        if save_predictions_path is not None:
            with open(save_predictions_path, "w", encoding="utf-8") as f:
                json.dump(final_predictions, f, indent=2)
            print("Saved predictions to:", save_predictions_path)

        return result, final_predictions

    finally:
        DENSE_WEIGHT = old_dense_weight
        BM25_WEIGHT = old_bm25_weight


def run_harmonic_grid_search(
    claims,
    dense_values=(250, 300),
    bm25_values=(150, 250),
    pool_values=(250, 300),
    top_k_values=(3, 5, 7),
    alpha_values=(0.6, 0.7, 0.8),
    dense_weight_values=(0.55, 0.65, 0.75),
    save_path="harmonic_grid_results.json"
):
    """
    Slower but more important than retrieval-only grid search:
    tune by final harmonic score, not just retrieval F1.
    """
    results = []

    for dense_n in dense_values:
        for bm25_n in bm25_values:
            for pool_size in pool_values:
                for top_k in top_k_values:
                    for alpha in alpha_values:
                        for dense_w in dense_weight_values:
                            bm25_w = 1.0 - dense_w

                            print("\n" + "=" * 80)
                            print(
                                f"Running harmonic setting: dense={dense_n}, bm25={bm25_n}, "
                                f"pool={pool_size}, k={top_k}, alpha={alpha}, "
                                f"dense_w={dense_w}, bm25_w={bm25_w}"
                            )

                            result, _ = run_end_to_end_setting(
                                claims,
                                dense_top_n=dense_n,
                                bm25_top_n=bm25_n,
                                candidate_pool_size=pool_size,
                                final_top_k=top_k,
                                alpha=alpha,
                                dense_weight=dense_w,
                                bm25_weight=bm25_w,
                                save_predictions_path=None
                            )

                            results.append(result)

                            results_sorted = sorted(
                                results,
                                key=lambda x: x["harmonic"],
                                reverse=True
                            )

                            with open(save_path, "w", encoding="utf-8") as f:
                                json.dump(results_sorted, f, indent=2)

                            print("\nCurrent best:")
                            print(results_sorted[0])

    results_sorted = sorted(results, key=lambda x: x["harmonic"], reverse=True)

    print("\n===== Top 10 harmonic settings =====")
    for row in results_sorted[:10]:
        print(row)

    return results_sorted


# Keep False by default because this reruns retrieval + classification many times.
RUN_HARMONIC_GRID_SEARCH = True

if RUN_HARMONIC_GRID_SEARCH:
    harmonic_results = run_harmonic_grid_search(
        dev_claims_small,
        dense_values=(250, 300),
        bm25_values=(150, 250),
        pool_values=(250, 300),
        top_k_values=(3, 5, 7),
        alpha_values=(0.6, 0.7, 0.8),
        dense_weight_values=(0.55, 0.65, 0.75),
        save_path="harmonic_grid_results.json"
    )
else:
    print("Set RUN_HARMONIC_GRID_SEARCH = True to tune directly for harmonic mean.")


# Recommended first manual trial after the default hybrid retrieval:
# result, preds = run_end_to_end_setting(
#     dev_claims_small,
#     dense_top_n=250,
#     bm25_top_n=250,
#     candidate_pool_size=300,
#     final_top_k=5,
#     alpha=0.75,
#     dense_weight=0.65,
#     bm25_weight=0.35,
#     save_predictions_path="dev_final_harmonic_tuned_predictions.json"
# )



Running harmonic setting: dense=250, bm25=150, pool=250, k=3, alpha=0.6, dense_w=0.55, bm25_w=0.44999999999999996


Hybrid retrieval + reranker: 100%|██████████| 154/154 [13:22<00:00,  5.21s/it]


===== End-to-end setting: dense=250, bm25=150, pool=250, k=3, alpha=0.6, dense_w=0.55, bm25_w=0.44999999999999996 =====
Retrieval Precision: 0.2078
Retrieval Recall:    0.2289
Retrieval F1:        0.2018
Zero-hit claims:     82 / 154
Zero-hit ratio:      0.5325
===== Official-style End-to-End Metrics =====
Evidence Retrieval F-score (F)    = 0.20176252319109464
Claim Classification Accuracy (A) = 0.4675324675324675
Harmonic Mean of F and A          = 0.2818802818802819

Current best:
{'dense_top_n': 250, 'bm25_top_n': 150, 'candidate_pool_size': 250, 'final_top_k': 3, 'alpha': 0.6, 'dense_weight': 0.55, 'bm25_weight': 0.44999999999999996, 'retrieval_precision': 0.2077922077922078, 'retrieval_recall': 0.2288961038961039, 'retrieval_f1': 0.20176252319109464, 'zero_hit_ratio': 0.5324675324675324, 'classification_accuracy': 0.4675324675324675, 'harmonic': 0.2818802818802819}

Running harmonic setting: dense=250, bm25=150, pool=250, k=3, alpha=0.6, dense_w=0.65, bm25_w=0.35


Hybrid retrieval + reranker: 100%|██████████| 154/154 [13:30<00:00,  5.26s/it]


===== End-to-end setting: dense=250, bm25=150, pool=250, k=3, alpha=0.6, dense_w=0.65, bm25_w=0.35 =====
Retrieval Precision: 0.2078
Retrieval Recall:    0.2289
Retrieval F1:        0.2018
Zero-hit claims:     82 / 154
Zero-hit ratio:      0.5325
===== Official-style End-to-End Metrics =====
Evidence Retrieval F-score (F)    = 0.20176252319109464
Claim Classification Accuracy (A) = 0.4675324675324675
Harmonic Mean of F and A          = 0.2818802818802819

Current best:
{'dense_top_n': 250, 'bm25_top_n': 150, 'candidate_pool_size': 250, 'final_top_k': 3, 'alpha': 0.6, 'dense_weight': 0.55, 'bm25_weight': 0.44999999999999996, 'retrieval_precision': 0.2077922077922078, 'retrieval_recall': 0.2288961038961039, 'retrieval_f1': 0.20176252319109464, 'zero_hit_ratio': 0.5324675324675324, 'classification_accuracy': 0.4675324675324675, 'harmonic': 0.2818802818802819}

Running harmonic setting: dense=250, bm25=150, pool=250, k=3, alpha=0.6, dense_w=0.75, bm25_w=0.25


...

===== End-to-end setting: dense=250, bm25=150, pool=250, k=5, alpha=0.6, dense_w=0.75, bm25_w=0.25 =====
Retrieval Precision: 0.1662
Retrieval Recall:    0.2963
Retrieval F1:        0.1984
Zero-hit claims:     71 / 154
Zero-hit ratio:      0.4610
===== Official-style End-to-End Metrics =====
Evidence Retrieval F-score (F)    = 0.19842815914244488
Claim Classification Accuracy (A) = 0.474025974025974
Harmonic Mean of F and A          = 0.27975172364096595

Current best:
{'dense_top_n': 250, 'bm25_top_n': 150, 'candidate_pool_size': 250, 'final_top_k': 3, 'alpha': 0.8, 'dense_weight': 0.55, 'bm25_weight': 0.44999999999999996, 'retrieval_precision': 0.20779220779220778, 'retrieval_recall': 0.23463203463203464, 'retrieval_f1': 0.2035095856524428, 'zero_hit_ratio': 0.538961038961039, 'classification_accuracy': 0.474025974025974, 'harmonic': 0.28476388636578487}

Running harmonic setting: dense=250, bm25=150, pool=250, k=5, alpha=0.7, dense_w=0.55, bm25_w=0.44999999999999996


Hybrid retrieval + reranker: 100%|██████████| 154/154 [10:53<00:00,  4.24s/it]


===== End-to-end setting: dense=250, bm25=150, pool=250, k=5, alpha=0.7, dense_w=0.55, bm25_w=0.44999999999999996 =====
Retrieval Precision: 0.1688
Retrieval Recall:    0.3062
Retrieval F1:        0.2031
Zero-hit claims:     68 / 154
Zero-hit ratio:      0.4416
===== Official-style End-to-End Metrics =====
Evidence Retrieval F-score (F)    = 0.20307153164296027
Claim Classification Accuracy (A) = 0.474025974025974
Harmonic Mean of F and A          = 0.2843347664939335

Current best:
{'dense_top_n': 250, 'bm25_top_n': 150, 'candidate_pool_size': 250, 'final_top_k': 3, 'alpha': 0.8, 'dense_weight': 0.55, 'bm25_weight': 0.44999999999999996, 'retrieval_precision': 0.20779220779220778, 'retrieval_recall': 0.23463203463203464, 'retrieval_f1': 0.2035095856524428, 'zero_hit_ratio': 0.538961038961039, 'classification_accuracy': 0.474025974025974, 'harmonic': 0.28476388636578487}

Running harmonic setting: dense=250, bm25=150, pool=250, k=5, alpha=0.7, dense_w=0.65, bm25_w=0.35


Hybrid retrieval + reranker:  35%|███▌      | 54/154 [05:28<10:08,  6.08s/it]


KeyboardInterrupt: 